# 4G LTE Throughput Prediction and Signal Quality Analysis

## Project Objective

This project investigates how radio-channel indicators and user-context
variables relate to downlink throughput in a 4G-focused mobile measurement
dataset. The main machine learning objective is to predict downlink throughput
using available radio and contextual features.

## Dataset Provenance

The dataset is based on the research dataset:

[**Beyond Throughput: a 4G LTE Dataset with Channel and Context Metrics**](https://doi.org/10.1145/3204949.3208123)

Authors: Darijo Raca, Jason J. Quinlan, Ahmed H. Zahran, and Cormac J. Sreenan  
Conference: ACM Multimedia Systems Conference, 2018  
DOI: `10.1145/3204949.3208123`

The dataset contains client-side mobile-network measurements collected from
two Irish operators under several mobility conditions. Although the dataset is
4G-focused, some traces also contain 2G and 3G observations. The exact modeling
scope will be decided after the data audit.

## Initial Scope

- Task type: Supervised machine learning
- Problem type: Regression
- Candidate target: `DL_bitrate` (kbit/s)
- Unit of observation: One timestamped network measurement
- Grouping unit: Measurement trace or session

## Important Limitation

The measurements were collected under specific operators, locations, devices,
and mobility scenarios. Therefore, conclusions from this project should not be
generalized automatically to all LTE networks.

For the telecom concepts behind the columns, see the companion
[domain notes](./README.md).

## Current Milestone — Discover the Files and Read One Trace

The goal of this first pass is intentionally small: confirm that the local data
is available, inventory the traces, open one CSV file, and understand what its
first and last observations look like. No modeling claim is made at this stage.

In [1]:
import sys

print(f"Python version: {sys.version.split()[0]}")

Python version: 3.12.13


In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

In [3]:
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
else:
    PROJECT_ROOT = CURRENT_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")

print("Project directories are configured successfully.")

Project directories are configured successfully.


In [4]:
csv_files = sorted(DATA_DIR.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError("No CSV files found in data directory")

print(f"Found {len(csv_files)} CSV files.")

Found 135 CSV files.


In [5]:
file_inventory = pd.DataFrame(
    {
        "File Name": [f.name for f in csv_files],
        "Relative Path": [f.relative_to(PROJECT_ROOT).as_posix() for f in csv_files],
        "Size (KiB)": [round(f.stat().st_size / 1024, 2) for f in csv_files],
    }
)

file_inventory_preview = file_inventory.head(10).copy()
file_inventory_preview.index = range(1, len(file_inventory_preview) + 1)

display(
    file_inventory_preview.style.format({"Size (KiB)": "{:,.2f}"})
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

,File Name,Relative Path,Size (KiB)
1,A_2017.11.30_16.48.26.csv,data/Dataset/bus/A_2017.11.30_16.48.26.csv,113.78
2,A_2018.01.25_16.33.53.csv,data/Dataset/bus/A_2018.01.25_16.33.53.csv,64.82
3,A_2018.01.25_17.27.30.csv,data/Dataset/bus/A_2018.01.25_17.27.30.csv,108.77
4,A_2018.01.25_18.02.07.csv,data/Dataset/bus/A_2018.01.25_18.02.07.csv,44.85
5,A_2018.01.25_19.50.40.csv,data/Dataset/bus/A_2018.01.25_19.50.40.csv,42.66
6,A_2018.01.26_11.26.26.csv,data/Dataset/bus/A_2018.01.26_11.26.26.csv,179.19
7,A_2018.01.27_10.58.49.csv,data/Dataset/bus/A_2018.01.27_10.58.49.csv,56.26
8,A_2018.01.27_11.12.23.csv,data/Dataset/bus/A_2018.01.27_11.12.23.csv,66.41
9,A_2018.01.27_12.10.00.csv,data/Dataset/bus/A_2018.01.27_12.10.00.csv,64.46
10,B_2018.01.25_16.33.45.csv,data/Dataset/bus/B_2018.01.25_16.33.45.csv,73.61


## Inspect One Trace

Before combining all 135 traces, I want to open one file and understand its
shape, columns, and observations. This keeps the first step concrete and makes
it easier to notice trace-level details that a merged table could hide.

In [6]:
sample_file = csv_files[0]

sample_df = pd.read_csv(sample_file)

print(f"Sample file: {sample_file.name}")
print(f"Trace shape: {sample_df.shape[0]:,} rows × {sample_df.shape[1]} columns")

Sample file: A_2017.11.30_16.48.26.csv
Trace shape: 910 rows × 20 columns


In [7]:
sample_path = sample_file.relative_to(PROJECT_ROOT).as_posix()
display(Markdown(f"**Sample trace:** `{sample_path}`"))

display(Markdown("### First 5 Rows"))
display(sample_df.head(5))

display(Markdown("### Last 5 Rows"))
display(sample_df.tail(5))

**Sample trace:** `data/Dataset/bus/A_2017.11.30_16.48.26.csv`

### First 5 Rows

,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate,UL_bitrate,State,NRxRSRP,NRxRSRQ,ServingCell_Lon,ServingCell_Lat,ServingCell_Distance
0,2017.11.30_16.48.26,-8.501373,51.893359,0,A,2,LTE,-102,-12,10.0,7,-85,3,7,D,-,-,-8.491719,51.893905,665.24000000000001
1,2017.11.30_16.48.26,-8.501291,51.893462,1,A,2,LTE,-102,-12,10.0,7,-85,3,7,D,-,-,-8.491719,51.893905,658.67999999999995
2,2017.11.30_16.48.27,-8.501291,51.893462,1,A,2,LTE,-102,-12,7.0,10,-87,310,14,D,-,-,-8.491719,51.893905,658.67999999999995
3,2017.11.30_16.48.28,-8.501291,51.893462,1,A,2,LTE,-102,-12,7.0,7,-85,0,0,I,-,-,-8.491719,51.893905,658.67999999999995
4,2017.11.30_16.48.29,-8.501291,51.893462,1,A,2,LTE,-102,-13,8.0,7,-85,0,0,I,-,-,-8.491719,51.893905,658.67999999999995


### Last 5 Rows

,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,CQI,RSSI,DL_bitrate,UL_bitrate,State,NRxRSRP,NRxRSRQ,ServingCell_Lon,ServingCell_Lat,ServingCell_Distance
905,2017.11.30_17.04.24,-8.556712,51.892251,46,A,1,LTE,-91,-12,3.0,12,-72,7068,128,D,-89.0,-10.0,-8.535593,51.880268,1968.8299999999999
906,2017.11.30_17.04.25,-8.556712,51.892251,46,A,1,LTE,-91,-12,3.0,10,-69,8992,166,D,-89.0,-10.0,-8.535593,51.880268,1968.8299999999999
907,2017.11.30_17.04.26,-8.556712,51.892251,46,A,1,LTE,-86,-11,5.0,9,-76,9584,177,D,-85.0,-9.0,-8.535593,51.880268,1968.8299999999999
908,2017.11.30_17.04.27,-8.556712,51.892251,46,A,1,LTE,-86,-11,5.0,10,-74,11060,198,D,-85.0,-9.0,-8.535593,51.880268,1968.8299999999999
909,2017.11.30_17.04.27,-8.556712,51.892251,46,A,1,LTE,-86,-11,5.0,5,-76,11060,198,D,-85.0,-9.0,-8.535593,51.880268,1968.8299999999999


### Current Checkpoint

This first pass confirms that the local CSV traces are discoverable and that one
selected trace can be read as a timestamped table with radio, context, and
bitrate measurements.

The next step is still data understanding: audit the schema across every trace,
identify non-numeric placeholders and suspicious values, and create an explicit
trace identifier before combining the files.